In [5]:
# === Robust Part 1 Runner (auto-detect local Input.xlsx) ===
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import re

# 0) Verify working dir and locate Input.xlsx
cwd = Path.cwd()
print("Working directory:", cwd)
print("Files here:", [p.name for p in cwd.iterdir()])

# Try to find Input.xlsx (case-insensitive) in current folder
candidates = [p for p in cwd.iterdir() if p.is_file() and p.suffix.lower() == ".xlsx" and p.stem.lower() == "input"]
if not candidates:
    raise FileNotFoundError("Couldn't find Input.xlsx in the current folder. "
                            "Either place it next to this notebook or set INPUT_PATH manually.")
INPUT_PATH = candidates[0]
print("Using:", INPUT_PATH)

# 1) Load
df = pd.read_excel(INPUT_PATH, sheet_name="Sheet1")  

# 2) Clean
def to_datetime_safe(series): return pd.to_datetime(series, errors="coerce")
def clean_text(s): return s if pd.isna(s) else str(s).strip()

for col in ["DateIssued", "RequiredByDate", "StatusDate"]:
    if col in df.columns:
        df[col] = to_datetime_safe(df[col])

for col in ["LOB", "ReqOrRec", "Status", "RiskImprovementTitle", "Client", "RiskEngineer", "Underwriter"]:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

for col in ["LOB", "ReqOrRec", "Status"]:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x.title() if isinstance(x, str) else x)

missing_summary = df.isnull().sum().rename("MissingCount").to_frame()
missing_summary["MissingPct"] = (missing_summary["MissingCount"] / len(df)).round(3)

# 3) Feature engineering
today = pd.Timestamp(datetime.today().date())

if {"StatusDate", "DateIssued"}.issubset(df.columns):
    df["DaysToComplete"] = (df["StatusDate"] - df["DateIssued"]).dt.days
else:
    df["DaysToComplete"] = np.nan

completed_like = {"Completed", "Closed", "Implemented", "Done"}
def is_open_status(status):
    if not isinstance(status, str): return True
    return status.title() not in completed_like

df["IsOpen"] = df["Status"].apply(is_open_status)

if "RequiredByDate" in df.columns:
    df["DaysOverdue"] = np.where(
        df["IsOpen"] & df["RequiredByDate"].notna(),
        (today - df["RequiredByDate"]).dt.days,
        0
    )
    df.loc[df["DaysOverdue"] < 0, "DaysOverdue"] = 0
else:
    df["DaysOverdue"] = np.nan

df["HasDeadline"] = df["RequiredByDate"].notna() if "RequiredByDate" in df.columns else False

if {"RequiredByDate", "DateIssued"}.issubset(df.columns):
    df["LeadTimeDays"] = (df["RequiredByDate"] - df["DateIssued"]).dt.days
else:
    df["LeadTimeDays"] = np.nan

def categorize_risk_title(title: str) -> str:
    if not isinstance(title, str) or not title.strip(): return "Other"
    t = title.lower()
    rules = [
        ("Fire Protection", r"\b(sprinkler|hydrant|fire alarm|extinguisher|fire door|suppression)\b"),
        ("Electrical", r"\b(electrical|wiring|breaker|switchgear|arc|thermography)\b"),
        ("Housekeeping", r"\b(housekeeping|storage|clutter|racking|pallet|waste)\b"),
        ("Security", r"\b(cctv|security|access control|intruder|fence|guard)\b"),
        ("Maintenance", r"\b(maintenance|inspection|service|repair|testing|preventive)\b"),
        ("Safety", r"\b(safety|ppe|guarding|signage|training)\b"),
        ("Business Continuity", r"\b(business continuity|bcp|resilience|continuity)\b"),
        ("Flood/Water", r"\b(flood|drain|sump|water ingress|leak|pump)\b"),
        ("Roof/Building Fabric", r"\b(roof|roofing|façade|facade|building fabric|cladding)\b"),
        ("Combustibles", r"\b(combustible|flammable|ignition|hot work|smoking)\b"),
    ]
    for label, pattern in rules:
        if re.search(pattern, t): return label
    return "Other"

if "RiskImprovementTitle" in df.columns:
    df["RiskCategory"] = df["RiskImprovementTitle"].apply(categorize_risk_title)
else:
    df["RiskCategory"] = "Other"

if "RiskEngineer" in df.columns:
    eng_open = df.groupby("RiskEngineer")["IsOpen"].sum().rename("EngineerLoad")
    df = df.merge(eng_open, on="RiskEngineer", how="left")
else:
    df["EngineerLoad"] = np.nan

if "Client" in df.columns:
    client_open = df.groupby("Client")["IsOpen"].sum().rename("ClientOpenImprovements")
    df = df.merge(client_open, on="Client", how="left")
else:
    df["ClientOpenImprovements"] = np.nan

# 4) Basic summaries (for sanity checks / Tableau)
status_counts = df["Status"].value_counts(dropna=False).rename_axis("Status").to_frame("Count")
lob_counts = df["LOB"].value_counts(dropna=False).rename_axis("LOB").to_frame("Count")
deadline_coverage = pd.DataFrame({
    "HasDeadline": ["Yes", "No"],
    "Count": [int(df["HasDeadline"].sum()), int((~df["HasDeadline"]).sum())]
})
deadline_coverage["Pct"] = (deadline_coverage["Count"] / len(df)).round(3)
lead_time_stats = df["LeadTimeDays"].dropna().describe().to_frame(name="LeadTimeDays_Stats")

# 5) Save next to your notebook
OUT_XLSX = cwd / "Cleaned_Risk_Improvement.xlsx"
OUT_CSV  = cwd / "Cleaned_Risk_Improvement.csv"

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="Cleaned", index=False)
    missing_summary.to_excel(writer, sheet_name="MissingSummary")
    status_counts.to_excel(writer, sheet_name="StatusCounts")
    lob_counts.to_excel(writer, sheet_name="LOBCounts")
    deadline_coverage.to_excel(writer, sheet_name="DeadlineCoverage", index=False)
    lead_time_stats.to_excel(writer, sheet_name="LeadTimeStats")

df.to_csv(OUT_CSV, index=False)

print("\nSUCCESS — Files created:")
print(" -", OUT_XLSX)
print(" -", OUT_CSV)


Working directory: /Users/tegamaseli/Documents/protector insurance presentation
Files here: ['Untitled-1.ipynb', 'Input.xlsx']
Using: /Users/tegamaseli/Documents/protector insurance presentation/Input.xlsx

SUCCESS — Files created:
 - /Users/tegamaseli/Documents/protector insurance presentation/Cleaned_Risk_Improvement.xlsx
 - /Users/tegamaseli/Documents/protector insurance presentation/Cleaned_Risk_Improvement.csv
